# DR Model Training after Artifact Removal

Matched ResNet18 experiment trained on the artifact-aware preprocessed images.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets,transforms,models
from torch.utils.data import DataLoader,random_split
BASE_DIR='/content/drive/MyDrive/Fundus_Artifact_Project'
DATA_DIR=os.path.join(BASE_DIR,'Results','dr_artifact_model','binary_dataset')
SAVE_DIR=os.path.join(BASE_DIR,'Results','dr_artifact_model')
os.makedirs(SAVE_DIR,exist_ok=True)


In [ ]:
IMG_SIZE=224; BATCH_SIZE=32; SEED=42
transform=transforms.Compose([transforms.Resize((IMG_SIZE,IMG_SIZE)),transforms.ToTensor(),transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
dataset=datasets.ImageFolder(DATA_DIR,transform=transform)
generator=torch.Generator().manual_seed(SEED)
n_train=int(0.8*len(dataset)); n_val=len(dataset)-n_train
train_set,val_set=random_split(dataset,[n_train,n_val],generator=generator)
train_loader=DataLoader(train_set,batch_size=BATCH_SIZE,shuffle=True)
val_loader=DataLoader(val_set,batch_size=BATCH_SIZE,shuffle=False)
print('Class mapping:',dataset.class_to_idx)


In [ ]:
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model=models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
model.fc=nn.Linear(model.fc.in_features,2); model=model.to(device)
criterion=nn.CrossEntropyLoss(); optimizer=optim.Adam(model.parameters(),lr=1e-4)
EPOCHS=10; history=[]
for epoch in range(EPOCHS):
    model.train(); tr_correct=tr_total=0
    for x,y in train_loader:
        x,y=x.to(device),y.to(device); optimizer.zero_grad(); out=model(x); loss=criterion(out,y); loss.backward(); optimizer.step()
        tr_correct+=(out.argmax(1)==y).sum().item(); tr_total+=y.size(0)
    model.eval(); va_correct=va_total=0
    with torch.no_grad():
        for x,y in val_loader:
            x,y=x.to(device),y.to(device); out=model(x); va_correct+=(out.argmax(1)==y).sum().item(); va_total+=y.size(0)
    row={'epoch':epoch+1,'train_accuracy':tr_correct/tr_total,'val_accuracy':va_correct/va_total}; history.append(row); print(row)
    torch.save(model.state_dict(),os.path.join(SAVE_DIR,f'epoch_{epoch+1}.pt'))
torch.save(model.state_dict(),os.path.join(SAVE_DIR,'dr_clean_resnet18.pt'))
pd.DataFrame(history).to_csv(os.path.join(SAVE_DIR,'clean_training_history.csv'),index=False)


## Matched comparison

Architecture, task definition, optimizer, image size, and nominal train/validation split settings are held as close as possible to the artifact-retaining baseline. The intended comparison isolates the preprocessing intervention rather than changing the model family.
